# 🇻🇳 Vietnamese Fake News: Crawling & Preprocessing Pipeline

**Complete workflow for Vietnamese news data collection and processing**

## 📋 Pipeline Overview

1. **Web Crawling** - Extract articles from Vietnamese news sites
2. **Data Preprocessing** - Process text (PhoBERT) and images (ResNet)
3. **Dataset Creation** - Generate ML-ready datasets
4. **Analysis & Export** - Visualize and save results


In [ ]:
# Setup
import sys, os, json, asyncio, nest_asyncio
import pandas as pd, numpy as np, torch
from pathlib import Path
from typing import List, Dict, Tuple
from datetime import datetime

warnings.filterwarnings("ignore")

nest_asyncio.apply()
project_root = Path().absolute().parent
sys.path.insert(0, str(project_root))

print(f"✅ Ready: {project_root}")

## 🕷️ Part 1: Web Crawling Configuration


In [ ]:
# Crawling setup
CRAWLING_CONFIG = {"max_pages": 10, "output_dir": "./crawled_data", "delay": 1}

TARGET_SITES = {
    "vnexpress": {"url": "https://vnexpress.net/", "crawler": "VnExpressCrawler"}
}

# Create directories
output_dir = Path(CRAWLING_CONFIG["output_dir"])
output_dir.mkdir(exist_ok=True)

print("🔧 Crawling configured")

In [ ]:
{
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "## 🕷️ Part 1: Web Crawling (Using CrawlerFactory)\n",
    "\n",
    "**Based on `src/test_crawler.py` - crawls from Hugging Face dataset**"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "# Set OPENSSL_CONF environment variable (required for crawler)\n",
    "import os\n",
    "os.environ[\"OPENSSL_CONF\"] = \"openssl.cnf\"\n",
    "\n",
    "# Import crawler modules\n",
    "try:\n",
    "    from src.crawler.crawler_factory import CrawlerFactory\n",
    "    from src.processing.dataset_handler import DatasetHandler\n",
    "    from src.helpers.logger import logger\n",
    "    print(\"✅ Crawler modules imported successfully\")\n",
    "except ImportError as e:\n",
    "    print(f\"❌ Error importing crawler modules: {e}\")\n",
    "    print(\"Make sure you're in the project root directory\")"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "# Crawling configuration (matching test_crawler.py)\n",
    "CRAWLING_CONFIG = {\n",
    "    \"dataset_name\": \"tranthaihoa/vifactcheck\",\n",
    "    \"url_column\": \"Url\",\n",
    "    \"splits\": [\"train\", \"test\", \"dev\"],\n",
    "    \"url_limit\": 5,  # Set to None for all URLs, small number for testing\n",
    "    \"cache_dir\": \"./data/caches\",\n",
    "    \"output_dir\": \"./src/data/json\"\n",
    "}\n",
    "\n",
    "# Create necessary directories\n",
    "os.makedirs(CRAWLING_CONFIG[\"cache_dir\"], exist_ok=True)\n",
    "os.makedirs(CRAWLING_CONFIG[\"output_dir\"], exist_ok=True)\n",
    "\n",
    "print(\"🔧 Crawling configured:\")\n",
    "for key, value in CRAWLING_CONFIG.items():\n",
    "    print(f\"  • {key}: {value}\")"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "async def crawl_vifactcheck_dataset():\n",
    "    \"\"\"Crawl URLs from VifactCheck dataset using CrawlerFactory\"\"\"\n",
    "    try:\n",
    "        dataset_name = CRAWLING_CONFIG[\"dataset_name\"]\n",
    "        url_column = CRAWLING_CONFIG[\"url_column\"]\n",
    "        splits = CRAWLING_CONFIG[\"splits\"]\n",
    "        url_limit = CRAWLING_CONFIG[\"url_limit\"]\n",
    "        \n",
    "        crawling_results = []\n",
    "        \n",
    "        for split in splits:\n",
    "            print(f\"\\n--- Processing split: {split} ---\")\n",
    "            \n",
    "            # Setup crawler factory\n",
    "            crawler_factory = CrawlerFactory(\n",
    "                cache_filename=f\"{CRAWLING_CONFIG['cache_dir']}/crawling_status_{split}.json\",\n",
    "                failed_log_filename=f\"{CRAWLING_CONFIG['cache_dir']}/failed_urls_{split}.json\"\n",
    "            )\n",
    "            \n",
    "            # Clear cache for fresh crawl (optional)\n",
    "            if crawler_factory.check_cache_file_exists():\n",
    "                print(f\"Clearing cache for split '{split}'...\")\n",
    "                crawler_factory.clear_cache()\n",
    "            \n",
    "            # Load URLs from dataset\n",
    "            dataset_handler = DatasetHandler(dataset_name)\n",
    "            urls_to_crawl = dataset_handler.get_urls_from_split(\n",
    "                split=split, url_column=url_column, limit=url_limit\n",
    "            )\n",
    "            \n",
    "            if urls_to_crawl:\n",
    "                print(f\"Found {len(urls_to_crawl)} URLs to crawl for {split}\")\n",
    "                \n",
    "                # Setup output filename\n",
    "                output_filename = f\"news_data_{dataset_name.split('/')[-1]}_{split}.json\"\n",
    "                output_path = os.path.join(CRAWLING_CONFIG[\"output_dir\"], output_filename)\n",
    "                \n",
    "                # Start crawling\n",
    "                print(f\"Starting crawl for {split} split...\")\n",
    "                await crawler_factory.crawl_and_save_all(urls_to_crawl, output_filename, format_name=\"custom\")\n",
    "                \n",
    "                result = {\n",
    "                    \"split\": split,\n",
    "                    \"urls_found\": len(urls_to_crawl),\n",
    "                    \"output_file\": output_path,\n",
    "                    \"timestamp\": datetime.now().isoformat()\n",
    "                }\n",
    "                crawling_results.append(result)\n",
    "                \n",
    "                print(f\"✅ Completed {split} split\")\n",
    "            else:\n",
    "                print(f\"❌ No URLs found for split {split}\")\n",
    "        \n",
    "        return crawling_results\n",
    "        \n",
    "    except Exception as e:\n",
    "        print(f\"❌ Crawling error: {e}\")\n",
    "        import traceback\n",
    "        traceback.print_exc()\n",
    "        return []\n",
    "\n",
    "# Test function definition\n",
    "print(\"🔧 Crawl function defined successfully\")"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "# Execute crawling (uncomment to run)\n",
    "# crawling_results = await crawl_vifactcheck_dataset()\n",
    "# \n",
    "# if crawling_results:\n",
    "#     print(\"\\n✅ Crawling completed!\")\n",
    "#     for result in crawling_results:\n",
    "#         print(f\"📁 {result['split']}: {result['urls_found']} URLs → {result['output_file']}\")\n",
    "# else:\n",
    "#     print(\"❌ No crawling results\")"
   ]
  }

## 🔄 Part 2: Data Preprocessing


In [ ]:
# Preprocessing config
PREPROCESSING_CONFIG = {
    "text_model": "vinai/phobert-base",
    "image_model": "resnet18",
    "language": "vi",
    "max_length": 512,
    "batch_size": 16,
    "save_format": "pkl",
}

# Device detection
device = (
    "cuda"
    if torch.cuda.is_available()
    else "mps" if torch.backends.mps.is_available() else "cpu"
)
print(f"🔧 Using device: {device}")

In [ ]:
# Initialize preprocessor
try:
    from src.preprocessing import CombinedPreprocessor

    preprocessor = CombinedPreprocessor(
        text_model_name=PREPROCESSING_CONFIG["text_model"],
        image_model_name=PREPROCESSING_CONFIG["image_model"],
        language=PREPROCESSING_CONFIG["language"],
        max_text_length=PREPROCESSING_CONFIG["max_length"],
        device=device,
    )
    print("✅ Preprocessor ready")
except Exception as e:
    print(f"❌ Preprocessor error: {e}")

In [ ]:
{
    "cell_type": "code",
    "execution_count": null,
    "metadata": {},
    "outputs": [],
    "source": [
        "def load_crawled_data():\n",
        '    """Load crawled data from the JSON output files"""\n',
        "    # Look for crawled data in the expected output directory\n",
        '    data_dir = Path(CRAWLING_CONFIG["output_dir"])\n',
        "    \n",
        "    if not data_dir.exists():\n",
        '        print(f"❌ Data directory not found: {data_dir}")\n',
        "        return []\n",
        "    \n",
        "    # Find all JSON files matching the pattern\n",
        '    json_files = list(data_dir.glob("news_data_vifactcheck_*.json"))\n',
        "    \n",
        "    all_articles = []\n",
        "    \n",
        "    for json_file in json_files:\n",
        "        try:\n",
        "            with open(json_file, 'r', encoding='utf-8') as f:\n",
        "                articles = json.load(f)\n",
        "            \n",
        "            # Add source information\n",
        "            for article in articles:\n",
        "                article['source_file'] = str(json_file)\n",
        "                article['source_split'] = json_file.stem.split('_')[-1]  # Extract split from filename\n",
        "            \n",
        "            all_articles.extend(articles)\n",
        '            print(f"✅ Loaded {len(articles)} articles from {json_file.name}")\n',
        "            \n",
        "        except Exception as e:\n",
        '            print(f"❌ Error loading {json_file}: {e}")\n',
        "    \n",
        '    print(f"\\n📊 Total articles loaded: {len(all_articles)}")\n',
        "    return all_articles\n",
        "\n",
        "# Load crawled data\n",
        "crawled_articles = load_crawled_data()\n",
        "\n",
        "if crawled_articles:\n",
        "    # Display sample article structure\n",
        '    print("\\n📄 Sample article structure:")\n',
        "    sample = crawled_articles[0]\n",
        "    for key, value in sample.items():\n",
        "        if isinstance(value, str) and len(value) > 100:\n",
        '            print(f"  • {key}: {value[:100]}...")\n',
        "        else:\n",
        '            print(f"  • {key}: {value}")',
    ],
}

In [ ]:
{
    "cell_type": "code",
    "execution_count": null,
    "metadata": {},
    "outputs": [],
    "source": [
        "def prepare_data_for_preprocessing(articles: List[Dict]) -> Tuple[List[str], List[str], List[int]]:\n",
        '    """\n',
        "    Prepare crawled VifactCheck data for preprocessing\n",
        "    \n",
        "    Returns:\n",
        "        Tuple of (texts, image_paths, labels)\n",
        '    """\n',
        "    texts = []\n",
        "    image_paths = []\n",
        "    labels = []\n",
        "    \n",
        "    for i, article in enumerate(articles):\n",
        "        # Extract text content (try multiple fields)\n",
        "        text = (article.get('text', '') or \n",
        "                article.get('content', '') or \n",
        "                article.get('title', '') or \n",
        "                article.get('description', ''))\n",
        "        \n",
        "        if text.strip():\n",
        "            texts.append(text.strip())\n",
        "            \n",
        "            # Extract image path\n",
        "            images = article.get('images', [])\n",
        "            if images and len(images) > 0:\n",
        "                # Handle different image formats\n",
        "                if isinstance(images[0], str):\n",
        "                    image_path = images[0]\n",
        "                elif isinstance(images[0], dict):\n",
        "                    image_path = (images[0].get('url', '') or \n",
        "                                images[0].get('path', '') or\n",
        "                                images[0].get('src', ''))\n",
        "                else:\n",
        "                    image_path = str(images[0])\n",
        "                \n",
        "                image_paths.append(image_path if image_path else None)\n",
        "            else:\n",
        "                image_paths.append(None)\n",
        "            \n",
        "            # Extract label (VifactCheck uses 'label' field)\n",
        "            label = article.get('label', 0)\n",
        "            if isinstance(label, str):\n",
        "                # Convert string labels to integers\n",
        "                label = 1 if label.lower() in ['fake', 'false', '1'] else 0\n",
        "            labels.append(int(label))\n",
        "    \n",
        "    return texts, image_paths, labels\n",
        "\n",
        "# Prepare data\n",
        "if crawled_articles:\n",
        "    texts, image_paths, labels = prepare_data_for_preprocessing(crawled_articles)\n",
        "    \n",
        '    print(f"📊 Data preparation completed:")\n',
        '    print(f"  • Text samples: {len(texts)}")\n',
        '    print(f"  • Image paths: {len(image_paths)}")\n',
        '    print(f"  • Labels: {len(labels)}")\n',
        '    print(f"  • Valid images: {sum(1 for path in image_paths if path is not None)}")\n',
        "    \n",
        "    if labels:\n",
        "        label_counts = {}\n",
        "        for label in labels:\n",
        "            label_counts[label] = label_counts.get(label, 0) + 1\n",
        '        print(f"  • Label distribution: {label_counts}")\n',
        "else:\n",
        '    print("❌ No crawled data available for preparation")',
    ],
}

In [ ]:
# Run preprocessing
if "preprocessor" in locals() and texts:
    try:
        print("🔄 Starting preprocessing...")

        # Create output directory
        processed_dir = "./processed_data/crawled"
        os.makedirs(processed_dir, exist_ok=True)

        # Process dataset
        dataset, metadata = preprocessor.preprocess_dataset(
            texts=texts,
            image_paths=processed_image_paths,
            labels=labels,
            save_dir=processed_dir,
            save_format=PREPROCESSING_CONFIG["save_format"],
            batch_size=PREPROCESSING_CONFIG["batch_size"],
        )

        print("✅ Preprocessing complete!")
        print(f"📊 Dataset: {len(dataset)} samples")
        print(f"📝 Text shape: {metadata['text_feature_shape']}")
        print(f"🖼️ Image shape: {metadata['image_feature_shape']}")

    except Exception as e:
        print(f"❌ Preprocessing error: {e}")
else:
    print("❌ Cannot preprocess - missing data or preprocessor")

## 📊 Part 3: Analysis & Visualization


In [ ]:
# Quick analysis
if "texts" in locals() and texts:
    import matplotlib.pyplot as plt
    from collections import Counter

    # Text statistics
    text_lengths = [len(text.split()) for text in texts]

    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

    # Text length distribution
    ax1.hist(text_lengths, bins=20, alpha=0.7)
    ax1.set_title("Text Length Distribution")
    ax1.set_xlabel("Words")
    ax1.set_ylabel("Frequency")

    # Label distribution
    label_counts = Counter(labels)
    ax2.bar(["Real", "Fake"], [label_counts.get(0, 0), label_counts.get(1, 0)])
    ax2.set_title("Label Distribution")
    ax2.set_ylabel("Count")

    plt.tight_layout()
    plt.show()

    print(f"📊 Total samples: {len(texts)}")
    print(f"📝 Avg text length: {np.mean(text_lengths):.1f} words")
    print(f"🏷️ Label distribution: {dict(label_counts)}")
else:
    print("❌ No data to analyze")

## 💾 Part 4: Export Results


In [ ]:
def export_results():
    """Export processed data in multiple formats"""
    try:
        processed_dir = "./processed_data/crawled"
        combined_path = os.path.join(processed_dir, "combined_dataset.pkl")

        if os.path.exists(combined_path):
            # Load data
            text_features, image_features, labels = preprocessor.load_combined_dataset(
                combined_path
            )

            # Export as NPZ
            npz_path = os.path.join(processed_dir, "dataset.npz")
            np.savez(
                npz_path,
                text_features=text_features,
                image_features=image_features,
                labels=labels,
            )

            # Export metadata CSV
            metadata_df = pd.DataFrame(
                {
                    "sample_id": range(len(labels)),
                    "label": labels,
                    "text_shape": [
                        text_features[i].shape for i in range(len(text_features))
                    ],
                }
            )

            csv_path = os.path.join(processed_dir, "metadata.csv")
            metadata_df.to_csv(csv_path, index=False)

            print(f"✅ Exported: NPZ ({npz_path}), CSV ({csv_path})")
        else:
            print("❌ No processed data found")

    except Exception as e:
        print(f"❌ Export error: {e}")


# Export results
export_results()

{
"cell*type": "code",
"execution_count": null,
"metadata": {},
"outputs": [],
"source": [
"# Pipeline status check\n",
"def check_status():\n",
" print(\"🚀 VifactCheck Pipeline Status:\")\n",
" print(\"=\" \* 40)\n",
" \n",
" # Check crawled data\n",
" data_dir = Path(CRAWLING_CONFIG[\"output_dir\"])\n",
" if data_dir.exists():\n",
" json_files = list(data_dir.glob(\"news_data_vifactcheck*_.json\"))\n",
" print(f\"📰 Crawled data: {len(json_files)} files found\")\n",
" for f in json_files:\n",
" print(f\" • {f.name}\")\n",
" else:\n",
" print(\"📰 Crawled data: Not found\")\n",
" \n",
" # Check cache files\n",
" cache_dir = Path(CRAWLING_CONFIG[\"cache_dir\"])\n",
" if cache_dir.exists():\n",
" cache_files = list(cache_dir.glob(\"_.json\"))\n",
" print(f\"💾 Cache files: {len(cache*files)} files\")\n",
" \n",
" # Check processed data\n",
" processed_dir = Path(\"./processed_data/crawled\")\n",
" if processed_dir.exists():\n",
" files = list(processed_dir.glob(\"\*\"))\n",
" print(f\"🔄 Processed data: {len(files)} files\")\n",
" else:\n",
" print(\"🔄 Processed data: Not found\")\n",
" \n",
" # Check preprocessor\n",
" if 'preprocessor' in locals():\n",
" print(\"✅ Preprocessor: Initialized\")\n",
" else:\n",
" print(\"❌ Preprocessor: Not initialized\")\n",
" \n",
" print(\"\\n🎯 Next steps:\")\n",
" print(\"1. Run crawling (uncomment crawling line)\")\n",
" print(\"2. Load and prepare data\")\n",
" print(\"3. Execute preprocessing with PhoBERT + ResNet\")\n",
" print(\"4. Analyze results and export for training\")\n",
" \n",
" print(\"\\n📁 Expected structure:\")\n",
" print(\" ./src/data/json/news_data_vifactcheck*_.json\")\n",
" print(\" ./data/caches/crawling*status*_.json\")\n",
" print(\" ./processed_data/crawled/\")\n",
"\n",
"check_status()"
]
}


In [ ]:
# Pipeline status check
def check_status():
    print("🚀 Pipeline Status:")
    print("=" * 40)

    # Check crawled data
    if output_dir.exists():
        json_files = list(output_dir.glob("**/*.json"))
        print(f"📰 Crawled: {len(json_files)} files")
    else:
        print("📰 Crawled: Not found")

    # Check processed data
    processed_dir = Path("./processed_data/crawled")
    if processed_dir.exists():
        files = list(processed_dir.glob("*"))
        print(f"🔄 Processed: {len(files)} files")
    else:
        print("🔄 Processed: Not found")

    print("\n🎯 Next steps:")
    print("1. Run crawling (uncomment line)")
    print("2. Execute preprocessing")
    print("3. Analyze results")
    print("4. Export for training")


check_status()